## 9 — LLM Resume Restructuring via Mistral

Sends the OCR-extracted text to an LLM via **Mistral** to dynamically detect
and restructure all resume sections into clean JSON.

- Model: `mistral-small-latest`
- API key loaded from `.env` as `MISTRAL_API_KEY`
- Uses `Mistral` client via mistralai SDK
- Sections are **detected dynamically** — no fixed schema assumed
- All content is preserved; nothing is discarded

1. Install `python-dotenv` and `mistralai`
2. Load Mistral API key and initialize Mistral client
3. Define the LLM parsing function
4. Run and save `resume_llm_structured.json`

---
## 1 — Install Dependencies


In [ ]:
%pip install mistralai pillow --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2 — Import Libraries


In [ ]:
import json          
import os            
from pathlib import Path  

from mistralai import Mistral  

print("All libraries imported successfully.")


All libraries imported successfully.


---
## 3 — Initialise the Mistral Client

The client is initialised with the API key and the OCR model name is defined here.
No language model (LLM) is used — only the OCR endpoint.


In [ ]:
from dotenv import load_dotenv


load_dotenv()


api_key = os.getenv('MISTRAL_API_KEY')

if not api_key:
    raise ValueError(
        'MISTRAL_API_KEY not found in .env file. '
        'Make sure your .env file exists with: MISTRAL_API_KEY=your_key'
    )


OCR_MODEL = "mistral-ocr-2512"


client = Mistral(api_key=api_key)

print("Mistral client initialised.")
print(f"  API key: {api_key[:10]}...{api_key[-4:]}" if api_key else "  API key: NOT SET")
print(f"  OCR model : {OCR_MODEL}")


Mistral client initialised.
  API key: kNVCjm7F0x...YmSX
  OCR model : mistral-ocr-2512


---
## 5 — OCR Extraction Function

`extract_text_with_mistral_ocr(file_path)` performs three steps:

1. **Upload** the document via `client.files.upload()`
2. **Get a signed URL** via `client.files.get_signed_url()`
3. **Call the OCR endpoint** via `client.ocr.process()` with `model="mistral-ocr-2512"`

Text is read from `response.pages[i].markdown` — no LLM is involved.


In [ ]:
def detect_file_type(file_path):
    """
    Detect the file type of a resume document from its extension.

    Parameters
    ----------
    file_path : str or Path

    Returns
    -------
    str  One of: 'pdf', 'png', 'jpg', 'jpeg'

    Raises
    ------
    FileNotFoundError : file does not exist
    ValueError        : extension is not supported
    """
    path = Path(file_path)

   
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

   
    ext = path.suffix.lower().lstrip(".")

    supported = {"pdf", "png", "jpg", "jpeg"}
    if ext not in supported:
        raise ValueError(
            f"Unsupported file type '.{ext}'. "
            f"Supported types: {', '.join(sorted(supported))}"
        )

    return ext


print("detect_file_type() defined.")


detect_file_type() defined.


---
## 5 — OCR Extraction Function

`extract_text_with_mistral_ocr(file_path)` performs three steps:

1. **Upload** the document via `client.files.upload()`
2. **Get a signed URL** via `client.files.get_signed_url()`
3. **Call the OCR endpoint** via `client.ocr.process()` with `model="mistral-ocr-2512"`

Text is read from `response.pages[i].markdown` — no LLM is involved.


In [ ]:
def extract_text_with_mistral_ocr(file_path):
    """
    Extract text from a PDF, PNG, JPG, or JPEG using Mistral OCR.

    Steps
    -----
    1. Validate file and detect type
    2. Upload file to Mistral Files API
    3. Retrieve a signed download URL
    4. Send the signed URL to the Mistral OCR endpoint
    5. Return the OCR response object (pages list + metadata)

    Parameters
    ----------
    file_path : str or Path

    Returns
    -------
    ocr_response  The raw OCR response from client.ocr.process()

    Raises
    ------
    FileNotFoundError : file does not exist
    ValueError        : unsupported file type
    RuntimeError      : upload, signed-URL, or OCR API call failed
    """
    path = Path(file_path)

    # Step 1 — detect file type (raises if unsupported or missing)
    file_type = detect_file_type(path)
    print(f"File detected: {path.name}  (type: {file_type})")

    
    print("Uploading file to Mistral Files API...")
    try:
        with open(path, "rb") as f:
            upload_response = client.files.upload(
                file={
                    "file_name": path.name,
                    "content": f,
                },
                purpose="ocr",  # mark the file for OCR use
            )
    except Exception as exc:
        raise RuntimeError(f"File upload failed: {exc}") from exc

    file_id = upload_response.id
    print(f"File uploaded successfully. File ID: {file_id}")

  
    print("Retrieving signed URL...")
    try:
        signed_url_response = client.files.get_signed_url(file_id=file_id)
    except Exception as exc:
        raise RuntimeError(f"Failed to get signed URL: {exc}") from exc

    signed_url = signed_url_response.url
    print("Signed URL retrieved.")

   
    print(f"Running OCR with model '{OCR_MODEL}'...")
    try:
        if file_type == "pdf":
            ocr_response = client.ocr.process(
                model=OCR_MODEL,
                document={
                    "type": "document_url",
                    "document_url": signed_url,
                },
            )
        else:
            # PNG, JPG, JPEG
            ocr_response = client.ocr.process(
                model=OCR_MODEL,
                document={
                    "type": "image_url",
                    "image_url": signed_url,
                },
            )
    except Exception as exc:
        raise RuntimeError(f"OCR API call failed: {exc}") from exc

    page_count = len(ocr_response.pages)
    print(f"OCR complete. Pages processed: {page_count}")

    return ocr_response


print("extract_text_with_mistral_ocr() defined.")


extract_text_with_mistral_ocr() defined.


---
## 6 — Combine Pages into One Text Block

`combine_pages(ocr_response)` iterates over `ocr_response.pages` and
concatenates the Markdown text from every page with a newline separator.


In [ ]:
def combine_pages(ocr_response):
    """
    Concatenate OCR text from all pages into a single string.

    Parameters
    ----------
    ocr_response  Return value of extract_text_with_mistral_ocr()

    Returns
    -------
    str  Full extracted text across all pages
    """
    full_text = ""

    for page in ocr_response.pages:
       
        full_text += page.markdown + "\n"

    return full_text.strip()


print("combine_pages() defined.")


combine_pages() defined.


---
## 7 — Export OCR Output to JSON

`export_to_json(file_path, ocr_response, output_path)` builds a structured
JSON object and writes it to `resume_ocr_output.json`.

```json
{
  "file_name": "resume.pdf",
  "page_count": 1,
  "pages": [
    {"page": 1, "text": "..."}
  ],
  "full_text": "..."
}
```


In [ ]:
def export_to_json(
    file_path,
    ocr_response,
    output_path="resume_ocr_output.json",
):
    """
    Build a structured JSON record from the OCR response and write it to disk.

    Parameters
    ----------
    file_path   : str or Path  Original source file
    ocr_response              Return value of extract_text_with_mistral_ocr()
    output_path : str          Destination JSON file (default: resume_ocr_output.json)

    Returns
    -------
    dict  The structured JSON object that was saved
    """
  
    pages_data = []
    for i, page in enumerate(ocr_response.pages, start=1):
        pages_data.append({
            "page": i,
            "text": page.markdown,
        })

  
    full_text = combine_pages(ocr_response)

   
    output = {
        "file_name": Path(file_path).name,
        "page_count": len(ocr_response.pages),
        "pages": pages_data,
        "full_text": full_text,
    }

    out_path = Path(output_path)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"JSON saved to: {out_path.resolve()}")
    return output


print("export_to_json() defined.")


export_to_json() defined.


---
## 8 — Example Usage

Set `file_path` to your resume file and run this cell.  
The pipeline runs automatically:
**upload → signed URL → OCR → combine pages → print → save JSON**


In [ ]:

file_path = r"C:\Users\HP\Downloads\cv3.pdf"   

print("[1/3] Extracting text with Mistral OCR...\n")
ocr_response = extract_text_with_mistral_ocr(file_path)


print("\n[2/3] Combining pages...")
text = combine_pages(ocr_response)


print("\n[3/3] Saving results...\n")
result = export_to_json(file_path, ocr_response, output_path="resume_ocr_output.json")

print("\n" + "=" * 70)
print("EXTRACTED TEXT")
print("=" * 70)
print(text)

print("\n" + "=" * 70)
print("JSON SUMMARY")
print("=" * 70)
summary = {
    "file_name": result["file_name"],
    "page_count": result["page_count"],
    "pages": [{"page": p["page"], "chars": len(p["text"])} for p in result["pages"]],
    "total_chars": len(result["full_text"]),
}
print(json.dumps(summary, indent=2))


[1/3] Extracting text with Mistral OCR...

File detected: cv3.pdf  (type: pdf)
Uploading file to Mistral Files API...
File uploaded successfully. File ID: cd033946-241c-4e52-beeb-7bf0f208b975
Retrieving signed URL...
Signed URL retrieved.
Running OCR with model 'mistral-ocr-2512'...
OCR complete. Pages processed: 1

[2/3] Combining pages...

[3/3] Saving results...

JSON saved to: C:\Users\HP\Documents\Career_assitant\agents\test\resume_ocr_output.json

EXTRACTED TEXT
John Doe

Data Scientist

John.doe@example.com

New York, USA

linkedin.com/in/johndoe

123-456-7890

https://www.johndoe.com

https://www.johndoe.com/portfolio

# SUMMARY

Highly motivated and experienced Data Scientist with a strong background in machine learning, data analysis, and database administration. Proven track record of delivering high-quality projects and driving business growth through data-driven insights.

# EXPERIENCE

## Senior Data Scientist

### ABC Corporation

Led a team of data scientists to develop

---
### Reference

| Topic | Detail |
|-------|--------|
| **OCR model** | `mistral-ocr-2512` — dedicated OCR endpoint, no LLM involved |
| **Upload** | `client.files.upload()` with `purpose="ocr"` |
| **Signed URL** | `client.files.get_signed_url()` — temporary access URL |
| **PDF** | Sent as `document_url` type to `client.ocr.process()` |
| **Image** | PNG / JPG / JPEG sent as `image_url` type |
| **Page text** | Read from `response.pages[i].markdown` |
| **Output file** | `resume_ocr_output.json` written next to the notebook |


---
## 9 — LLM Resume Restructuring via OpenRouter

Sends the OCR-extracted text to an LLM via **OpenRouter** to dynamically detect
and restructure all resume sections into clean JSON.

- Model: `openai/gpt-4o-mini`
- API key loaded from `.env` as `OPENROUTER_API_KEY`
- Uses `ChatOpenAI` via langchain-openai
- Sections are **detected dynamically** — no fixed schema assumed
- All content is preserved; nothing is discarded

1. Install `python-dotenv` and `langchain-openai`
2. Load OpenRouter API key and initialize ChatOpenAI
2. Load OpenRouter API key
3. Define the LLM parsing function
4. Run and save `resume_llm_structured.json`


In [ ]:

%pip install python-dotenv mistralai --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
from mistralai import Mistral

print("LLM step libraries imported.")

LLM step libraries imported.


In [ ]:

load_dotenv(override=True)  

MISTRAL_API_KEY = (os.getenv("MISTRAL_API_KEY") or "").strip().strip('"').strip("'")

if not MISTRAL_API_KEY:
    raise ValueError(
        "MISTRAL_API_KEY not found. "
        "Create a .env file in this directory containing:\n"
        "  MISTRAL_API_KEY=your_key_here"
    )

MISTRAL_MODEL = "mistral-small-latest"


client = Mistral(api_key=MISTRAL_API_KEY)

print("Mistral config loaded.")
print(f"  Model : {MISTRAL_MODEL}")
print(f"  Key   : {MISTRAL_API_KEY[:10]}...{MISTRAL_API_KEY[-4:]}")

Mistral config loaded.
  Model : mistral-small-latest
  Key   : kNVCjm7F0x...YmSX


In [ ]:
def call_mistral(prompt):
    """
    Send a prompt to the Mistral API and return the response.

    Parameters
    ----------
    prompt : str

    Returns
    -------
    str  Raw text returned by the model

    Raises
    ------
    RuntimeError : API call failed
    """
    try:
        response = client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    except Exception as exc:
        raise RuntimeError(f"Mistral API call failed: {exc}") from exc


def extract_json(raw):
    """
    Parse a JSON object from the LLM response string.

    Strips markdown code fences if the model included them despite
    being told not to.

    Parameters
    ----------
    raw : str

    Returns
    -------
    dict

    Raises
    ------
    ValueError : response cannot be parsed as JSON
    """
 
    cleaned = re.sub(r'^```[a-zA-Z]*\s*', '', raw, flags=re.MULTILINE)
    cleaned = re.sub(r'\s*```$', '', cleaned, flags=re.MULTILINE).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "LLM response is not valid JSON.\n"
            f"First 500 chars:\n{raw[:500]}"
        ) from exc


def build_prompt(ocr_text):
    """
    Build the LLM prompt that instructs dynamic, lossless resume parsing.

    The model is told to:
    - Detect ALL sections dynamically (no fixed schema)
    - Preserve every piece of information verbatim
    - Return only a valid JSON object (no markdown fences, no commentary)
    """
    return (
        "You are an expert resume parser.\n\n"
        "Read the resume text below and convert it into a clean, structured "
        "JSON object.\n\n"
        "RULES YOU MUST FOLLOW:\n"
        "1. Detect sections DYNAMICALLY from the content — do NOT assume a "
        "fixed structure.\n"
        "2. Include every section you find, including uncommon ones "
        "(e.g. Strengths, Certifications, Languages, Awards, Publications, etc.).\n"
        "3. Preserve all information exactly as written — do NOT summarise, "
        "rephrase, or omit anything.\n"
        "4. Use null for missing scalar fields and [] for missing arrays — "
        "never invent data.\n"
        "5. Return ONLY a valid JSON object — no markdown fences, no "
        "explanation, no extra text whatsoever.\n\n"
        "OUTPUT STRUCTURE GUIDELINES:\n"
        "- Top-level keys: name, contact, sections\n"
        "- contact: { email, phone, location, linkedin, website, ... } "
        "(include whatever is present)\n"
        "- sections: object whose keys are the detected section names in "
        "snake_case (e.g. experience, education, skills, projects, "
        "certifications, languages, strengths, ...)\n"
        "- Experience / education entries: { title, organisation, location, "
        "start_date, end_date, description, bullets[] }\n"
        "- Project entries: { title, role, start_date, end_date, description, "
        "technologies[] }\n"
        "- Simple list sections (skills, languages, etc.): array of strings\n\n"
        f"RESUME TEXT:\n{ocr_text}\n\n"
        "Return only the JSON object:"
    )


def parse_resume_with_llm(ocr_text, output_path="resume_llm_structured.json"):
    """
    Full pipeline: prompt → Mistral → parse JSON → save to disk.

    Parameters
    ----------
    ocr_text    : str   Raw OCR text from combine_pages() or resume_ocr_output.json
    output_path : str   Destination file for the structured JSON

    Returns
    -------
    dict  Structured resume as returned by the LLM
    """
    if not ocr_text or not ocr_text.strip():
        raise ValueError(
            "OCR text is empty. Run the OCR extraction cells first."
        )

   
    print("[1/3] Sending OCR text to Mistral LLM...")
    prompt = build_prompt(ocr_text)
    raw = call_mistral(prompt)
    print(f"      Response received ({len(raw)} chars).")

    print("[2/3] Parsing JSON from LLM response...")
    structured = extract_json(raw)
    print(f"      Top-level keys: {list(structured.keys())}")


    print(f"[3/3] Saving to '{output_path}'...")
    out = Path(output_path)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(structured, f, indent=2, ensure_ascii=False)
    print(f"      Saved → {out.resolve()}")

    return structured


print("LLM parsing functions defined (build_prompt / call_mistral / extract_json / parse_resume_with_llm).")

LLM parsing functions defined (build_prompt / call_mistral / extract_json / parse_resume_with_llm).


---
### Run the LLM Step

Uses the `text` variable set by cell 8 (`combine_pages`) if available.
Falls back to reading `resume_ocr_output.json` from disk if the notebook
was not run top-to-bottom in this session.

Output is saved to **`resume_llm_structured.json`**.


In [ ]:

try:
    _ocr_text = text
    print(f"Using in-memory OCR text ({len(_ocr_text)} chars).")
except NameError:
    _ocr_file = Path("resume_ocr_output.json")
    if not _ocr_file.exists():
        raise FileNotFoundError(
            "resume_ocr_output.json not found. "
            "Run the OCR extraction cell (cell 8) first."
        )
    with open(_ocr_file, "r", encoding="utf-8") as _f:
        _ocr_text = json.load(_f)["full_text"]
    print(f"Loaded OCR text from disk ({len(_ocr_text)} chars).")

print()


llm_structured_resume = parse_resume_with_llm(
    _ocr_text,
    output_path="resume_llm_structured.json",
)


print("\n" + "=" * 70)
print("LLM STRUCTURED RESUME")
print("=" * 70)

display(Markdown(
    "```json\n"
    + json.dumps(llm_structured_resume, indent=2, ensure_ascii=False)
    + "\n```"
))


Using in-memory OCR text (2922 chars).

[1/3] Sending OCR text to Mistral LLM...
      Response received (4142 chars).
[2/3] Parsing JSON from LLM response...
      Top-level keys: ['name', 'contact', 'sections']
[3/3] Saving to 'resume_llm_structured.json'...
      Saved → C:\Users\HP\Documents\Career_assitant\agents\test\resume_llm_structured.json

LLM STRUCTURED RESUME


```json
{
  "name": "John Doe",
  "contact": {
    "email": "John.doe@example.com",
    "location": "New York, USA",
    "linkedin": "linkedin.com/in/johndoe",
    "phone": "123-456-7890",
    "website": "https://www.johndoe.com",
    "portfolio": "https://www.johndoe.com/portfolio"
  },
  "sections": {
    "summary": "Highly motivated and experienced Data Scientist with a strong background in machine learning, data analysis, and database administration. Proven track record of delivering high-quality projects and driving business growth through data-driven insights.",
    "experience": [
      {
        "title": "Senior Data Scientist",
        "organisation": "ABC Corporation",
        "location": "New York, USA",
        "start_date": "Jan 2020",
        "end_date": "Present",
        "description": "Led a team of data scientists to develop and implement predictive models that increased sales by 25% and reduced costs by 15%. Collaborated with cross-functional teams to design and deploy data pipelines, resulting in a 30% improvement in data quality and a 40% reduction in data processing time.",
        "bullets": [
          "Developed and deployed machine learning models using Python, R, and SQL",
          "Designed and implemented data pipelines using Apache Beam, Apache Spark, and AWS Glue",
          "Collaborated with data engineers to optimize database performance and improve data storage"
        ]
      },
      {
        "title": "Data Scientist",
        "organisation": "DEF Startups",
        "location": "San Francisco, USA",
        "start_date": "Jun 2018",
        "end_date": "Dec 2019",
        "description": "Worked on multiple projects, including customer segmentation, churn prediction, and recommendation systems. Developed and deployed models that resulted in a 20% increase in customer engagement and a 15% reduction in customer churn.",
        "bullets": [
          "Developed and deployed machine learning models using Python, R, and SQL",
          "Collaborated with data engineers to design and implement data pipelines",
          "Worked with business stakeholders to identify opportunities and develop data-driven solutions"
        ]
      }
    ],
    "projects": [
      {
        "title": "Customer Segmentation",
        "role": "Lead Data Scientist",
        "start_date": "Mar 2020",
        "end_date": "Jun 2020",
        "description": "Developed a customer segmentation model using clustering algorithms that resulted in a 25% increase in targeted marketing campaigns.",
        "technologies": [
          "Python",
          "R",
          "SQL",
          "Apache Spark"
        ]
      },
      {
        "title": "Churn Prediction",
        "role": "Data Scientist",
        "start_date": "Sep 2019",
        "end_date": "Dec 2019",
        "description": "Developed a churn prediction model using machine learning algorithms that resulted in a 20% reduction in customer churn.",
        "technologies": [
          "Python",
          "R",
          "SQL",
          "Apache Spark"
        ]
      }
    ],
    "skills": [
      "Machine Learning",
      "Data Analysis",
      "Database Administration",
      "Python",
      "R",
      "SQL",
      "Apache Spark",
      "Communication",
      "Team Management"
    ],
    "languages": [
      "English",
      "Spanish"
    ],
    "education": [
      {
        "title": "Master's",
        "organisation": "Stanford University",
        "location": "Stanford, USA",
        "start_date": null,
        "end_date": "Jun 2018",
        "description": "Data Science • 3.8",
        "bullets": []
      }
    ],
    "certificates": [
      {
        "title": "Certified Data Scientist",
        "organisation": "Data Science Council of America",
        "start_date": null,
        "end_date": "Dec 2019",
        "description": "Certified data scientist with expertise in machine learning, data analysis, and database administration.",
        "bullets": []
      }
    ],
    "strengths": [
      "Strong analytical and problem-solving skills",
      "Excellent communication and teamwork skills"
    ]
  }
}
```